In [ ]:
import pandas as pd
from transformers import T5ForConditionalGeneration, T5Tokenizer
from sklearn.model_selection import train_test_split
import torch
from transformers import get_scheduler

# Load the dataset from an Excel file
df = pd.read_excel("/content/sample_data/First_1000_Patent_Samples.xlsx")

# Ensure there are no nulls in key columns
df = df.dropna(subset=['abstract', 'claims'])

# Selecting 'claims' as input for summarization and 'abstract' to keep the same
X = df['claims']
Y = df['abstract']

# Splitting the data
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

# Load the T5 model and tokenizer
model = T5ForConditionalGeneration.from_pretrained("t5-base")
tokenizer = T5Tokenizer.from_pretrained("t5-base")

# Setup for training
train_dataset = [("summarize: " + str(x), y) for x, y in zip(X_train, Y_train)]
val_dataset = [("summarize: " + str(x), y) for x, y in zip(X_val, Y_val)]
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

# Setup the learning rate scheduler
num_training_steps = 3 * len(train_dataset)  # Adjust epochs and training length accordingly
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# Training loop
model.train()
num_epochs = 3  # Adjust based on performance and computational resources
for epoch in range(num_epochs):
    for x, y in train_dataset:
        optimizer.zero_grad()
        input_ids = tokenizer(x, return_tensors='pt', padding=True, truncation=True, max_length=512).input_ids
        labels = tokenizer(y, return_tensors='pt', padding=True, truncation=True, max_length=128).input_ids
        outputs = model(input_ids, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()  # Update the learning rate after each batch

    # Validation loss calculation
    model.eval()
    total_loss = 0
    for x, y in val_dataset:
        with torch.no_grad():
            input_ids = tokenizer(x[0], return_tensors='pt', padding=True, truncation=True, max_length=512).input_ids
            labels = tokenizer(x[1], return_tensors='pt', padding=True, truncation=True, max_length=128).input_ids
            outputs = model(input_ids, labels=labels)
            total_loss += outputs.loss.item()
    val_loss = total_loss / len(val_dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Validation Loss: {val_loss}")

# Generate and store summaries for the claims in the DataFrame
df['generated_summary'] = ''
model.eval()
with torch.no_grad():
    for index, row in df.iterrows():
        input_claim = row['claims']
        input_ids = tokenizer("summarize: " + input_claim, return_tensors='pt', padding=True, truncation=True, max_length=512).input_ids
        output_ids = model.generate(input_ids, max_length=200, num_beams=4, early_stopping=True)
        summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        df.at[index, 'generated_summary'] = summary

# Save the updated DataFrame with the summaries to an Excel file
df.to_csv("/content/sample_data/First_1000_Samples_summaries.csv", index=False)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Epoch 1/3, Validation Loss: 4.514536380767822
Epoch 2/3, Validation Loss: 4.468481540679932
Epoch 3/3, Validation Loss: 4.456296443939209


In [ ]:
pip install pandas torch bert_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.1 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.19.3-py3-none-manylinux1_x86_64.whl (166.0 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manyli

In [ ]:
import pandas as pd
import torch
import re
from bert_score import score

# Load the DataFrame from the Excel file
# Load the DataFrame from the CSV file
input_file = "/content/sample_data/First_1000_Samples_summaries.csv"  # Modify the path as needed
df = pd.read_csv(input_file)


# Define a list to store the BERT metric scores
bert_metric_scores = []

# Set device to GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Iterate over the rows in the DataFrame
for index, row in df.iterrows():
    # Clean the text to remove non-ASCII characters
    abstract = re.sub(r'[^\x00-\x7F]+', '', str(row['abstract']))
    generated_summary = str(row['generated_summary'])

    # Calculate BERT metric scores
    _, _, bert_metric_score = score([generated_summary], [abstract], model_type="bert-base-uncased", device=device)

    # Append the score to the BERT metric scores list
    bert_metric_scores.append(bert_metric_score.item())

# Add the scores to the DataFrame
df["BERT_Metric_Score"] = bert_metric_scores

# Save the updated DataFrame to a new Excel file
output_file = "/content/sample_data/First_1000_Samples_bert_score.csv"
df.to_csv(output_file, index=False)

# Print the average BERT metric score
print("Average BERT Metric Score:", sum(bert_metric_scores) / len(bert_metric_scores))

Average BERT Metric Score: 0.7059675816595554


In [ ]:
pip install pandas rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24933 sha256=7dad4e31972f5da26bc16aae911763b8ba482e26475239450f1961bf3e61d37a
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score


In [ ]:
import pandas as pd
import re
from rouge_score import rouge_scorer

# Load the DataFrame from the Excel file
input_file = "/content/sample_data/First_1000_Samples_summaries.csv"  # Modify the path as needed
df = pd.read_csv(input_file)

# Define a list to store the ROUGE scores
rouge_scores = []

# Create a ROUGE scorer instance for ROUGE-1, ROUGE-2, and ROUGE-L
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Iterate over the rows in the DataFrame
for index, row in df.iterrows():
    # Clean the text to remove non-ASCII characters
    abstract = re.sub(r'[^\x00-\x7F]+', '', str(row['abstract']))
    generated_summary = str(row['generated_summary'])

    # Calculate ROUGE scores
    scores = scorer.score(abstract, generated_summary)

    # Append the scores dictionary to the ROUGE scores list
    rouge_scores.append(scores)

# Add the ROUGE scores to the DataFrame
df['ROUGE_Scores'] = rouge_scores

# Save the updated DataFrame to a new Excel file
output_file = "/content/sample_data/First_1000_Samples_rouge_score.csv"
df.to_csv(output_file, index=False)

# Optionally, print the average scores for each ROUGE metric
average_scores = {metric: 0 for metric in ['rouge1', 'rouge2', 'rougeL']}
for scores in rouge_scores:
    for key in average_scores:
        average_scores[key] += scores[key].fmeasure
for key in average_scores:
    average_scores[key] /= len(rouge_scores)

print("Average ROUGE Scores:", average_scores)


Average ROUGE Scores: {'rouge1': 0.5824088773541851, 'rouge2': 0.39078600414796955, 'rougeL': 0.4588373910423154}


In [ ]:
pip install pandas torch transformers tqdm sklearn numpy

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
import pandas as pd
from tqdm import tqdm
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
import re
import numpy as np

# Load the DataFrame from the Excel file
input_file = "/content/sample_data/First_1000_Samples_summaries.csv"  # Modify the path as needed
df = pd.read_csv(input_file)

# Define a list to store SummaC scores
summac_scores = []

# Initialize BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Set device to GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def get_bert_embeddings(text):
    """Generate BERT embeddings for the given text."""
    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(device)
    outputs = model(**inputs)
    # Use the pooled output directly to represent the whole sequence
    return outputs['pooler_output'].detach().cpu().numpy()

# Process each row in the DataFrame
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Calculating Scores"):
    abstract = re.sub(r'[^\x00-\x7F]+', ' ', str(row['abstract']))  # Clean non-ASCII characters
    original_text = abstract
    generated_summary = str(row['generated_summary'])

    # Get embeddings for original text and generated summary
    original_text_embedding = get_bert_embeddings(original_text)
    generated_summary_embedding = get_bert_embeddings(generated_summary)

    # Calculate cosine similarity between embeddings
    cos_sim = cosine_similarity(original_text_embedding, generated_summary_embedding)[0][0]
    summac_scores.append(cos_sim)

# Add the scores to the DataFrame
df['SummaC_Score'] = summac_scores

# Save the updated DataFrame to a new Excel file
output_file = "/content/sample_data/First_1000_Samples_summac_score.csv"
df.to_csv(output_file, index=False)
# Print the average SummaC score
print("Average SummaC Score:", np.mean(summac_scores))


Calculating Scores: 100%|██████████| 1000/1000 [00:30<00:00, 32.54it/s]


Average SummaC Score: 0.92225087
